# 4차시 실습: AI 죄수의 딜레마 — 우리 팀 AI 준비

게임에서 우리 팀 대신 협상할 AI(언어 모델)를 Colab의 GPU에 올립니다. **팀에서 한 명만** 실행하면 됩니다.

1. 아래 **① AI 준비** 셀의 ▶ 를 누릅니다. 5~8분 걸립니다.
2. 끝에 나오는 `https://....trycloudflare.com` 주소를 복사합니다.
3. `게임접속.html` 을 열고 **AI 주소** 칸에 붙여 넣습니다.

이 탭을 닫으면 AI도 꺼집니다. 게임이 끝날 때까지 열어 두세요.

> "GPU 없음"이 나오면: 메뉴 **런타임 > 런타임 유형 변경 > T4 GPU > 저장** 후 셀을 다시 실행합니다.

In [ ]:
#@title ① AI 준비 (▶ 누르고 5~8분) { display-mode: "form" }
MODEL = "gemma4:e4b"  #@param ["gemma4:e4b", "gemma4:12b"]
import json
import os
import re
import shutil
import subprocess
import time
import urllib.request

OLLAMA = "http://127.0.0.1:11434"
TUNNEL_LOG = "/content/cloudflared.log"
CLOUDFLARED = "/content/cloudflared"
CLOUDFLARED_URL = ("https://github.com/cloudflare/cloudflared/releases/latest/"
                   "download/cloudflared-linux-amd64")


def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)


def _get(url, timeout=5):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return resp.read().decode("utf-8")


def _post(path, body, timeout=600):
    req = urllib.request.Request(
        OLLAMA + path, data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def gpu_name():
    r = _sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
    return r.stdout.strip() if r.returncode == 0 else ""


def ollama_alive():
    try:
        _get(OLLAMA + "/api/tags", timeout=2)
        return True
    except Exception:
        return False


def install_ollama():
    if shutil.which("ollama"):
        return
    _sh("apt-get -qq install -y zstd pciutils")
    r = _sh("curl -fsSL https://ollama.com/install.sh | sh")
    if not shutil.which("ollama"):
        raise RuntimeError("Ollama 설치 실패: " + (r.stderr or r.stdout)[-400:])


def start_ollama(parallel=2):
    # OLLAMA_HOST=0.0.0.0 과 OLLAMA_ORIGINS=* 가 있어야 게임 화면(브라우저)이 터널 너머에서 호출할 수 있다.
    if ollama_alive():
        return
    env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_ORIGINS="*",
               OLLAMA_KEEP_ALIVE="60m", OLLAMA_NUM_PARALLEL=str(parallel))
    subprocess.Popen(["ollama", "serve"], env=env,
                     stdout=open("/content/ollama.log", "a"),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        if ollama_alive():
            return
        time.sleep(1)
    raise RuntimeError("Ollama 서버가 시작되지 않았습니다. /content/ollama.log 를 확인하세요.")


def pull_model(model, show=True):
    req = urllib.request.Request(
        OLLAMA + "/api/pull", data=json.dumps({"model": model}).encode("utf-8"),
        headers={"Content-Type": "application/json"})
    biggest, shown = 0, -20
    with urllib.request.urlopen(req, timeout=3600) as resp:
        for line in resp:
            ev = json.loads(line)
            if ev.get("error"):
                raise RuntimeError("모델 받기 실패: " + ev["error"])
            total, done = ev.get("total", 0), ev.get("completed", 0)
            if total >= biggest and total > 0:
                biggest = total
                pct = int(done * 100 / total)
                if show and pct >= shown + 20:
                    shown = pct
                    print(f"   모델 받는 중 {pct:3d}%  ({total / 1e9:.1f}GB)", flush=True)


def warm_up(model):
    # 모델을 GPU 메모리에 올리고 짧은 답을 한 번 받아 둔다.
    # T4에서는 올린 뒤 첫 답만 1분쯤 걸리므로, 첫 통화가 늦지 않게 여기서 미리 치른다.
    _post("/api/chat", {"model": model, "stream": False, "keep_alive": "60m",
                        "options": {"num_predict": 8},
                        "messages": [{"role": "user", "content": "안녕"}]})


def _tunnel_url_in_log():
    if not os.path.exists(TUNNEL_LOG):
        return ""
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUNNEL_LOG).read())
    return m.group(0) if m else ""


def start_tunnel():
    running = _sh("pgrep -f 'cloudflared tunnel'").stdout.strip()
    if running and _tunnel_url_in_log():
        return _tunnel_url_in_log()
    if not os.path.exists(CLOUDFLARED):
        urllib.request.urlretrieve(CLOUDFLARED_URL, CLOUDFLARED)
        os.chmod(CLOUDFLARED, 0o755)
    open(TUNNEL_LOG, "w").close()
    subprocess.Popen(
        [CLOUDFLARED, "tunnel", "--url", OLLAMA,
         "--http-host-header", "127.0.0.1:11434", "--no-autoupdate"],
        stdout=open(TUNNEL_LOG, "a"), stderr=subprocess.STDOUT, start_new_session=True)
    url = ""
    for _ in range(60):
        url = _tunnel_url_in_log()
        if url:
            break
        time.sleep(1)
    if not url:
        raise RuntimeError("임시 주소를 만들지 못했습니다. 셀을 다시 실행하세요.")
    # 주소가 실제로 열릴 때까지 기다린다(보통 5~20초).
    for _ in range(40):
        try:
            _get(url + "/api/tags", timeout=5)
            return url
        except Exception:
            time.sleep(2)
    raise RuntimeError("임시 주소가 응답하지 않습니다. 셀을 다시 실행하세요.")


def prepare(model, parallel=2):
    t0 = time.time()
    gpu = gpu_name()
    if gpu:
        print(f"1/5 GPU 확인: {gpu}")
    else:
        print("1/5 GPU 없음. [런타임] > [런타임 유형 변경] > T4 GPU 로 바꾼 뒤 다시 실행하세요.")
        print("    (GPU 없이도 진행은 되지만 AI 답이 매우 느립니다.)")
    print("2/5 Ollama 설치 중... (약 30초)", flush=True)
    install_ollama()
    start_ollama(parallel)
    print(f"3/5 모델 {model} 받는 중... (2~6분)", flush=True)
    pull_model(model)
    print("4/5 모델을 GPU에 올리는 중... (2~3분)", flush=True)
    warm_up(model)
    print("5/5 임시 주소 만드는 중...", flush=True)
    url = start_tunnel()
    print(f"\n준비 끝 ({int(time.time() - t0)}초). 아래 주소를 복사해 게임 화면의 [AI 주소] 칸에 붙여 넣으세요.\n")
    print("   " + url + "\n")
    print("이 탭을 닫으면 주소가 사라집니다. 게임이 끝날 때까지 열어 두세요.")
    return url

URL = prepare(MODEL)

In [ ]:
#@title ② 주소 다시 보기 · 상태 확인 { display-mode: "form" }
print("AI 주소:", start_tunnel())
print("GPU    :", gpu_name() or "없음")
loaded = json.loads(_get(OLLAMA + "/api/ps"))["models"]
for m in loaded:
    print(f"모델   : {m['name']}  (GPU 메모리 {m.get('size_vram', 0) / 1e9:.1f}GB 사용)")
if not loaded:
    print("모델   : 아직 GPU에 올라가 있지 않습니다. ① 셀을 다시 실행하세요.")

In [ ]:
#@title ③ AI에게 한마디 (동작 확인) { display-mode: "form" }
question = "안녕하세요. 한 문장으로 자기소개 해 주세요."  #@param {type: "string"}
t = time.time()
answer = _post("/api/chat", {"model": MODEL, "stream": False,
                             "messages": [{"role": "user", "content": question}]})
print(answer["message"]["content"].strip())
print(f"\n({time.time() - t:.1f}초)")